# Surprise Housing India 🇮🇳 - Real Estate Investment & Valuation Model
## Advanced Regression Modeling & Predictive Analytics for Indian Metropolitan Markets

### Executive Summary
Surprise Housing is expanding its real estate investment portfolio into major Indian metropolitan markets (Mumbai, Bengaluru, Delhi NCR, Hyderabad, Pune, Chennai, Kolkata). This notebook implements an end-to-end Machine Learning pipeline to predict property valuations in INR (₹), evaluate regularization techniques (**Lasso** and **Ridge**), train ensemble algorithms (**Random Forest** and **XGBoost**), and extract critical property drivers (RERA compliance, Vastu, Metro proximity, BHK, Carpet area) to guide strategic acquisition decisions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to sys.path
sys.path.append('..')

from generate_dataset import generate_surprise_housing_data
from src.preprocessing import preprocess_housing_data
from src.models import (
    train_and_tune_lasso,
    train_and_tune_ridge,
    train_random_forest,
    train_xgboost,
    evaluate_all_models,
    extract_feature_importance
)

sns.set_theme(style='darkgrid')
print('All libraries imported successfully.')

### Step 1: Indian Housing Data Exploration & Preprocessing
We generate and inspect the Indian housing dataset, analyzing feature distributions, carpet area scaling, RERA status, and target variable skewness (`SalePrice_INR`). Logarithmic transformation `np.log1p(SalePrice_INR)` is applied to satisfy regression normality assumptions.

In [ ]:
# Load or generate dataset
df = generate_surprise_housing_data(n_samples=2000)
print(f'Dataset dimensions: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

### Step 2: Feature Engineering & Robust Scaling
We engineer domain-specific features:
- `Area_Efficiency_Ratio`: Carpet Area / Super Built-up Area
- `Floor_Ratio`: Floor Number / Total Building Floors
- `Amenity_Score`: Sum of Gated Society, Power Backup, Clubhouse, Parking, RERA Approval

Strict ML featurization ordering is enforced: train-test split is executed **BEFORE** fitting `RobustScaler` to eliminate data leakage risks.

In [ ]:
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw, scaler, feature_names = preprocess_housing_data(df)
print(f'Training Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples')
print(f'Total Encoded Features: {len(feature_names)}')

### Step 3: Model Training & Hyperparameter Tuning
We train 4 candidate models:
1. **Lasso Regression (L1 Regularization)**: Tunes alpha via 5-Fold CV to enforce feature sparsity.
2. **Ridge Regression (L2 Regularization)**: Tunes alpha via 5-Fold CV to handle collinearity.
3. **Random Forest Regressor**: Ensembles decision trees to capture non-linear interactions.
4. **XGBoost Regressor**: Gradient boosting algorithm for high predictive accuracy.

In [ ]:
print('Training & Tuning Lasso...')
lasso_model, lasso_alpha, lasso_cv = train_and_tune_lasso(X_train, y_train_log)

print('Training & Tuning Ridge...')
ridge_model, ridge_alpha, ridge_cv = train_and_tune_ridge(X_train, y_train_log)

print('Training Random Forest...')
rf_model = train_random_forest(X_train, y_train_log)

print('Training XGBoost...')
xgb_model = train_xgboost(X_train, y_train_log)

models = {
    'Lasso Regression': lasso_model,
    'Ridge Regression': ridge_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}
print('All 4 models trained successfully.')

### Step 4: Model Evaluation in INR (₹) Scale
We evaluate models on the test set after inverse log transformation `np.expm1()`.

In [ ]:
evaluation = evaluate_all_models(models, X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw)
eval_df = pd.DataFrame(evaluation).T[['Train_R2', 'Test_R2', 'Test_MAE', 'Test_RMSE', 'Test_MAPE']]
eval_df['Test_MAE_Lakhs'] = eval_df['Test_MAE'] / 100000
eval_df

### Step 5: Key Value Drivers (Lasso Feature Importances)
Extracting top positive and negative house price drivers in the Indian market.

In [ ]:
lasso_pos = extract_feature_importance(lasso_model, feature_names).sort_values(ascending=False).head(10)
lasso_neg = extract_feature_importance(lasso_model, feature_names).sort_values(ascending=True).head(5)

plt.figure(figsize=(10, 5))
sns.barplot(x=lasso_pos.values, y=lasso_pos.index, palette='Greens_r')
plt.title('Top 10 Positive Property Value Drivers (Lasso)')
plt.xlabel('Lasso Coefficient')
plt.show()